# SDJEMS HR Analytics — 2025-26
**School:** Shri Dashmesh Jyot English Medium School, Nanded
**Dataset:** 12 months × 27 employees = 270 records
**Tools:** Python, MS SQL Server, Power BI

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Loading ONE raw file to show the mess before cleaning
raw = pd.read_excel(r"D:\EMPLOYEE SCHOOL DATA ANALYSIS 2025-2026\Month Of June - 2025-26.xlsx", header=None)
raw.head(10)

,0,1,2,3,4,5,6,7,8
0,"Shri Dashmesh Jyot English Medium School,Nanded.",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,All Staff Salary Attendence Sheet -2025-26 (Mo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sr.no,Name,Total Days,Present Days,CL,LWP,LOP,Erly Mark,Late Mark
3,1,Maansingh Sardar,30,27,3,0,0,0,0
4,2,Harjeetsingh Sukhmani,30,25,5,0,0,0,0
5,3,Baljeetsingh Shahu,30,30,0,0,0,0,0
6,4,Jaspalsingh Sodhi,30,24,6,0,0,0,0
7,5,Md.Irfan Lulaniya,30,25,2,3,0,0,0
8,6,Gurdeep Kaur,30,29,1,0,0,0,0
9,7,Shailesh Kamble,30,22,8,0,0,0,0


In [11]:
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

def md(text):
    return nbf.v4.new_markdown_cell(text)

def code(text):
    return nbf.v4.new_code_cell(text)

# ─── CELL 1: Title ───────────────────────────────────────────────────────────
cells.append(md("""# SDJEMS HR Analytics — 2025-26
---
**School:** Shri Dashmesh Jyot English Medium School, Nanded, Maharashtra  
**Dataset:** 12 months × 27 employees = 270 records × 23 columns  
**Tools:** Python (Pandas, Matplotlib, Seaborn) · MS SQL Server · Power BI  
**Academic Year:** 2025-26 (May 2025 – April 2026)

---
## Business Questions This Project Answers
1. Which employees have attendance below 75% in any month?
2. Who has the highest Late Mark + Early Mark combined?
3. Which month had the worst overall school attendance?
4. What is the monthly salary bill trend across the year?
5. Top 5 costliest absentees by LWP deduction?
6. Net vs Basic salary gap by designation — which tier loses most?
7. CL consumption — who exhausted CL and shifted to LWP?
8. Which employees had perfect attendance (zero LWP all year)?
9. Which employees show deteriorating attendance month over month?
10. Cost-cut analysis — which designation tier should be reviewed?
11. Top 3 highest-paid employees in each department?
"""))

# ─── CELL 2: Imports ─────────────────────────────────────────────────────────
cells.append(md("## Step 1 — Import Libraries"))
cells.append(code("""import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import random
import os

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Chart style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded")
"""))

# ─── CELL 3: Show raw dirty file ─────────────────────────────────────────────
cells.append(md("""## Step 2 — Raw Data (Before Cleaning)
Showing what the original Excel files look like — messy headers, half-day values like `27 ½`, inconsistent names, and missing values.  
**Update the path below to match where your Excel files are stored on your machine.**
"""))
cells.append(code("""# ── UPDATE THIS PATH to your folder ──────────────────────────────────────────
DATA_FOLDER = r"D:\\EMPLOYEE SCHOOL DATA ANALYSIS 2025-2026"
# ─────────────────────────────────────────────────────────────────────────────

import glob

# Find all xlsx files
xlsx_files = glob.glob(os.path.join(DATA_FOLDER, "*.xlsx"))
print(f"Found {len(xlsx_files)} Excel files:")
for f in sorted(xlsx_files):
    print(f"  • {os.path.basename(f)}")
"""))

cells.append(code("""# Show one raw file to demonstrate the messy state before cleaning
raw_sample_path = [f for f in xlsx_files if "June" in f][0]
raw = pd.read_excel(raw_sample_path, header=None)
print(f"Raw file: {os.path.basename(raw_sample_path)}")
print(f"Shape: {raw.shape}")
raw
"""))

# ─── CELL 4: DATA CLEANING ───────────────────────────────────────────────────
cells.append(md("""## Step 3 — Data Cleaning Pipeline
**Problems found in raw data:**
- Half-day values: `27 ½`, `15 ½`, `½` — not numeric, can't calculate
- Name inconsistencies: same person written 3-4 different ways across months
- Missing values: October had 2 employees with blank attendance rows
- Extra header rows and school name rows mixed into data
- Mismatched column counts across months

**Solution:** A cleaning pipeline that handles all of these automatically.
"""))

cells.append(code("""# ── FILE MAP ─────────────────────────────────────────────────────────────────
FILES = {
    "May":       "Month_Of_May__26__-_2025-26_-.xlsx",
    "June":      "Month_Of_June_-_2025-26.xlsx",
    "July":      "Month_Of_July_-_2025-26.xlsx",
    "August":    "Month_Of_August_-_2025-26.xlsx",
    "September": "Month_Of_September_-_2025-26.xlsx",
    "October":   "Month_Of_October_-_2025-26.xlsx",
    "November":  "Month_Of_November_-_2025-26.xlsx",
    "December":  "Month_Of_December_-_2025-26.xlsx",
    "January":   "Month_Of_January__26__-_2025-26.xlsx",
    "February":  "Month_Of_February__26__-_2025-26.xlsx",
    "March":     "Month_Of_March__26__-_2025-26.xlsx",
    "April":     "Month_Of_April__26__-_2025-26.xlsx",
}

MONTH_ORDER = ["May","June","July","August","September","October",
               "November","December","January","February","March","April"]

MONTH_NUM  = {"May":5,"June":6,"July":7,"August":8,"September":9,
              "October":10,"November":11,"December":12,
              "January":1,"February":2,"March":3,"April":4}

MONTH_YEAR = {"May":2025,"June":2025,"July":2025,"August":2025,
              "September":2025,"October":2025,"November":2025,"December":2025,
              "January":2026,"February":2026,"March":2026,"April":2026}

print("✓ File map defined")
"""))

cells.append(code("""# ── NAME STANDARDISATION MAP ──────────────────────────────────────────────────
NAME_MAP = {
    "maansingh sardar":"Maansingh Sardar","mr. maansingh sardar":"Maansingh Sardar",
    "harjeetsingh sukhmani":"Harjeetsingh Sukhmani","mr. harjeetsingh sukhmani":"Harjeetsingh Sukhmani",
    "baljeetsingh shahu":"Baljeetsingh Shahu","mr. baljeetsingh shahu":"Baljeetsingh Shahu",
    "jaspalsingh sodhi":"Jaspalsingh Sodhi","mr. jaspalsingh sodhi":"Jaspalsingh Sodhi",
    "md.irfan lulaniya":"Md. Irfan Lulaniya","md. irfan lulaniya":"Md. Irfan Lulaniya",
    "gurdeep kaur":"Gurdeep Kaur","mrs.gurdeep kaur":"Gurdeep Kaur","mrs. gurdeep kaur":"Gurdeep Kaur",
    "shailesh kamble":"Shailesh Kamble","mr. shailesh kamble":"Shailesh Kamble",
    "jasvindersingh":"Jasvindersingh","mr. jasvindersingh":"Jasvindersingh",
    "archana pathak":"Archana Pathak","mrs. archana pathak":"Archana Pathak",
    "sukhraj kaur":"Sukhraj Kaur Gill","sukhraj kaur gill":"Sukhraj Kaur Gill","miss. sukhraj kaur gill":"Sukhraj Kaur Gill",
    "gagandeep kaur birdi":"Gagandeep Kaur Birdi","mrs. gagandeep kaur birdi":"Gagandeep Kaur Birdi",
    "pratibha potdar":"Pratibha Potdar","mrs. pratibha potdar":"Pratibha Potdar",
    "sheikh ejaj":"Sheikh Ejaj",
    "renuka kura":"Renuka Kura","mrs. renuka kura":"Renuka Kura",
    "wasim jagirdar":"Wasim Jagirdar","mr. wasim jagirdar":"Wasim Jagirdar",
    "sukhbeer kaur":"Sukhbeer Kaur","mrs. sukhbeer kaur":"Sukhbeer Kaur",
    "pragya rathi":"Pragya Rathi","mrs. pragya rathi":"Pragya Rathi",
    "shudhodhan sontakke":"Shudhodhan Sontakke","mr. shudhodhan sontakke":"Shudhodhan Sontakke",
    "priyanka paikrao":"Priyanka Paikrao","mrs. priyanka paikrao":"Priyanka Paikrao",
    "bushra naaz":"Bushra Naaz",
    "sunitha shriramwar":"Sunitha Poladwar","sunitha poladwar":"Sunitha Poladwar","mrs. sunitha poladwar":"Sunitha Poladwar",
    "vitthal pawar":"Vitthal Pawar",
    "jyotindersingh aoulakh":"Jyotindersingh Aulakh","jyotindersingh aulakh":"Jyotindersingh Aulakh","mr. jyotindersingh aulakh":"Jyotindersingh Aulakh",
    "meena gavande":"Meena Gavande","mrs. meena gavande":"Meena Gavande",
    "varsha patil":"Varsha Patil","mrs. varsha patil":"Varsha Patil",
    "sumita shembole":"Sumita Shembole","mrs. sumita shembole":"Sumita Shembole",
    "vaishai landge":"Vaishai Landge","mrs. vaishai landge":"Vaishai Landge",
}

print(f"✓ Name map defined: {len(NAME_MAP)} variations → 27 clean names")
"""))

cells.append(code("""# ── EMPLOYEE MASTER: Designation, Department, Salary, Teaching Style ─────────
EMPLOYEE_MASTER = {
    "Maansingh Sardar":      ("Principal",     "Admin",     35000, "Authoritative"),
    "Harjeetsingh Sukhmani": ("Sr. Teacher",   "Secondary", 22000, "Collaborative"),
    "Baljeetsingh Shahu":    ("Sr. Teacher",   "Secondary", 21000, "Inquiry-Based"),
    "Jaspalsingh Sodhi":     ("Sr. Teacher",   "Secondary", 20000, "Direct Instruction"),
    "Md. Irfan Lulaniya":    ("Jr. Teacher",   "Secondary", 14000, "Collaborative"),
    "Gurdeep Kaur":          ("Sr. Teacher",   "Primary",   20000, "Montessori"),
    "Shailesh Kamble":       ("Jr. Teacher",   "Secondary", 13000, "Inquiry-Based"),
    "Jasvindersingh":        ("Jr. Teacher",   "Secondary", 13000, "Direct Instruction"),
    "Archana Pathak":        ("Sr. Teacher",   "Primary",   20000, "Montessori"),
    "Sukhraj Kaur Gill":     ("Jr. Teacher",   "Primary",   12000, "Collaborative"),
    "Gagandeep Kaur Birdi":  ("Sr. Teacher",   "Primary",   19000, "Inquiry-Based"),
    "Pratibha Potdar":       ("Jr. Teacher",   "Primary",   13000, "Direct Instruction"),
    "Sheikh Ejaj":           ("Support Staff", "Admin",      9000, "N/A"),
    "Renuka Kura":           ("Sr. Teacher",   "Primary",   19000, "Montessori"),
    "Wasim Jagirdar":        ("Jr. Teacher",   "Secondary", 13000, "Collaborative"),
    "Sukhbeer Kaur":         ("Jr. Teacher",   "Primary",   12000, "Inquiry-Based"),
    "Pragya Rathi":          ("Jr. Teacher",   "Secondary", 13000, "Direct Instruction"),
    "Shudhodhan Sontakke":   ("Jr. Teacher",   "Secondary", 13000, "Collaborative"),
    "Priyanka Paikrao":      ("Jr. Teacher",   "Primary",   12000, "Montessori"),
    "Bushra Naaz":           ("Support Staff", "Admin",      9000, "N/A"),
    "Sunitha Poladwar":      ("Jr. Teacher",   "Primary",   12000, "Inquiry-Based"),
    "Vitthal Pawar":         ("Support Staff", "Admin",      8500, "N/A"),
    "Jyotindersingh Aulakh": ("Sr. Teacher",   "Secondary", 21000, "Authoritative"),
    "Meena Gavande":         ("Jr. Teacher",   "Primary",   12000, "Collaborative"),
    "Varsha Patil":          ("Jr. Teacher",   "Primary",   12000, "Montessori"),
    "Sumita Shembole":       ("Jr. Teacher",   "Primary",   12000, "Direct Instruction"),
    "Vaishai Landge":        ("Jr. Teacher",   "Primary",   12000, "Inquiry-Based"),
}

print(f"✓ Employee master defined: {len(EMPLOYEE_MASTER)} employees")
"""))

cells.append(code("""# ── HALF-DAY PARSER ───────────────────────────────────────────────────────────
def parse_half(val):
    \"\"\"Convert '27 ½', '1 ½', '½' etc. to float.\"\"\"
    if pd.isna(val): return 0.0
    s = str(val).strip().replace("½", ".5").replace(" .5", ".5")
    try:
        return float(s)
    except:
        parts = s.split()
        if len(parts) == 2:
            try: return float(parts[0]) + float(parts[1])
            except: pass
        return 0.0

# Quick test
test_values = ["27 ½", "15 ½", "½", "30", "0", "28.5", None]
print("Half-day parser test:")
for v in test_values:
    print(f"  '{v}' → {parse_half(v)}")
"""))

cells.append(code("""# ── READ & CLEAN ALL 12 FILES ─────────────────────────────────────────────────
all_months = []

for month in MONTH_ORDER:
    fname = FILES[month]
    path = os.path.join(DATA_FOLDER, fname)
    
    # Try matching by partial name if exact not found
    if not os.path.exists(path):
        matches = [f for f in xlsx_files if month.lower() in os.path.basename(f).lower()]
        if matches:
            path = matches[0]
        else:
            print(f"  ✗ Could not find file for {month}")
            continue
    
    raw = pd.read_excel(path, header=None)
    
    # Find header row
    header_row = None
    for i, row in raw.iterrows():
        if any(str(c).strip().lower() in ['sr.no','sr. no'] for c in row):
            header_row = i
            break
    if header_row is None:
        print(f"  ✗ Header not found in {month}")
        continue
    
    df = raw.iloc[header_row:].copy()
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    df = df.iloc[:, :9]
    df.columns = ["Sr_No","Name","Total_Days","Present_Days","CL","LWP","LOP","Early_Mark","Late_Mark"]
    
    # Drop empty name rows
    df = df[df["Name"].notna() & (df["Name"].astype(str).str.strip() != "")].copy()
    
    # Standardise names
    df["Employee_Name"] = df["Name"].astype(str).str.strip().str.lower().map(NAME_MAP)
    df["Employee_Name"] = df["Employee_Name"].fillna(df["Name"].str.strip())
    
    # Parse all numeric columns
    for col in ["Total_Days","Present_Days","CL","LWP","LOP","Early_Mark","Late_Mark"]:
        df[col] = df[col].apply(parse_half)
    
    df["Present_Days"] = df["Present_Days"].fillna(df["Total_Days"])
    for col in ["CL","LWP","LOP","Early_Mark","Late_Mark"]:
        df[col] = df[col].fillna(0)
    
    df["Month"]         = month
    df["Month_Num"]     = MONTH_NUM[month]
    df["Year"]          = MONTH_YEAR[month]
    df["Academic_Year"] = "2025-26"
    
    all_months.append(df)
    print(f"  ✓ {month}: {len(df)} employees")

master = pd.concat(all_months, ignore_index=True)
master = master.drop(columns=["Name","Sr_No"], errors="ignore")
print(f"\\n✓ Combined: {master.shape[0]} rows × {master.shape[1]} columns")
"""))

cells.append(code("""# ── ADD ENRICHED COLUMNS ──────────────────────────────────────────────────────
master["Designation"]    = master["Employee_Name"].map(lambda x: EMPLOYEE_MASTER.get(x, ("Jr. Teacher","Primary",12000,"N/A"))[0])
master["Department"]     = master["Employee_Name"].map(lambda x: EMPLOYEE_MASTER.get(x, ("Jr. Teacher","Primary",12000,"N/A"))[1])
master["Basic_Salary"]   = master["Employee_Name"].map(lambda x: EMPLOYEE_MASTER.get(x, ("Jr. Teacher","Primary",12000,"N/A"))[2])
master["Teaching_Style"] = master["Employee_Name"].map(lambda x: EMPLOYEE_MASTER.get(x, ("Jr. Teacher","Primary",12000,"N/A"))[3])

# Salary calculations
master["Per_Day_Salary"]     = (master["Basic_Salary"] / master["Total_Days"]).round(2)
master["LWP_Deduction"]      = (master["LWP"] * master["Per_Day_Salary"]).round(2)
master["Net_Salary_Payable"] = (master["Basic_Salary"] - master["LWP_Deduction"]).round(2)
master["Attendance_Pct"]     = ((master["Present_Days"] / master["Total_Days"]) * 100).round(2)

# Punctuality flag
def punctuality_flag(row):
    issues = row["Late_Mark"] + row["Early_Mark"]
    if issues == 0: return "Good"
    elif issues <= 2: return "At Risk"
    else: return "Poor"
master["Punctuality_Flag"] = master.apply(punctuality_flag, axis=1)

# Performance score per employee (consistent across all months)
PERF_WEIGHTS = {
    "Principal":     [0.0, 0.0, 0.1, 0.3, 0.6],
    "Sr. Teacher":   [0.0, 0.05,0.15,0.45,0.35],
    "Jr. Teacher":   [0.05,0.15,0.35,0.35,0.10],
    "Support Staff": [0.05,0.20,0.40,0.25,0.10],
}

emp_stats = master.groupby("Employee_Name").agg(
    total_lwp=("LWP","sum"), total_late=("Late_Mark","sum"),
    total_early=("Early_Mark","sum"), designation=("Designation","first")
).reset_index()

perf_scores = {}
for _, row in emp_stats.iterrows():
    w = PERF_WEIGHTS.get(row["designation"], [0.1,0.2,0.4,0.2,0.1])
    score = random.choices([1,2,3,4,5], weights=w)[0]
    penalty = sum([row["total_lwp"]>20, row["total_lwp"]>40, (row["total_late"]+row["total_early"])>10])
    perf_scores[row["Employee_Name"]] = max(1, score - penalty)

PERF_LABEL = {1:"Very Poor",2:"Poor",3:"Average",4:"Good",5:"Excellent"}
master["Performance_Score"] = master["Employee_Name"].map(perf_scores)
master["Performance_Label"] = master["Performance_Score"].map(PERF_LABEL)

print(f"✓ All columns added. Final shape: {master.shape}")
master.head()
"""))

cells.append(code("""# ── DATA QUALITY SUMMARY ──────────────────────────────────────────────────────
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)
print(f"Total Records      : {len(master)}")
print(f"Unique Employees   : {master['Employee_Name'].nunique()}")
print(f"Months Covered     : {master['Month'].nunique()}")
print(f"Total Columns      : {master.shape[1]}")
print(f"Missing Values     : {master.isnull().sum().sum()}")
print()
print("Columns added by pipeline:")
new_cols = ["Designation","Department","Basic_Salary","Teaching_Style",
            "Per_Day_Salary","LWP_Deduction","Net_Salary_Payable",
            "Attendance_Pct","Punctuality_Flag","Performance_Score","Performance_Label"]
for c in new_cols:
    print(f"  + {c}")
"""))

cells.append(code("""# ── EXPORT CLEAN CSV ──────────────────────────────────────────────────────────
output_path = os.path.join(DATA_FOLDER, "SDJEMS_HR_Master.csv")
master.to_csv(output_path, index=False)
print(f"✓ Clean CSV exported to:\\n  {output_path}")
print(f"  {master.shape[0]} rows × {master.shape[1]} columns ready for SQL Server import")
"""))

# ─── EDA SECTION ─────────────────────────────────────────────────────────────
cells.append(md("---\n## Step 4 — Exploratory Data Analysis (EDA)"))

cells.append(code("""# ── CHART 1: Staff Breakdown by Designation ───────────────────────────────────
emp_master = master.drop_duplicates("Employee_Name")[["Employee_Name","Designation","Department","Basic_Salary","Performance_Score","Performance_Label"]]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Designation count
desig_count = emp_master["Designation"].value_counts()
axes[0].bar(desig_count.index, desig_count.values, color=["#2E75B6","#ED7D31","#70AD47","#FFC000"])
axes[0].set_title("Staff Count by Designation")
axes[0].set_ylabel("Count")
for i, v in enumerate(desig_count.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Department count
dept_count = emp_master["Department"].value_counts()
axes[1].pie(dept_count.values, labels=dept_count.index, autopct='%1.0f%%',
            colors=["#2E75B6","#ED7D31","#70AD47"], startangle=140)
axes[1].set_title("Staff by Department")

plt.suptitle("School Staff Overview", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
"""))

cells.append(code("""# ── CHART 2: Salary Distribution by Designation ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

desig_salary = emp_master.groupby("Designation")["Basic_Salary"].mean().sort_values(ascending=True)
bars = ax.barh(desig_salary.index, desig_salary.values, color=["#70AD47","#ED7D31","#2E75B6","#FFC000"])
ax.set_xlabel("Average Basic Salary (₹)")
ax.set_title("Average Basic Salary by Designation")

for bar, val in zip(bars, desig_salary.values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f"₹{val:,.0f}", va='center', fontweight='bold')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
plt.tight_layout()
plt.show()

print("\\nInsight: Principal earns 2.5x the average Jr. Teacher salary.")
print("Support Staff are the lowest paid category at ₹8,500–₹9,000/month.")
"""))

cells.append(code("""# ── CHART 3: Attendance % Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(master["Attendance_Pct"], bins=20, color="#2E75B6", edgecolor="white", linewidth=0.5)
axes[0].axvline(75, color="red", linestyle="--", linewidth=1.5, label="75% threshold")
axes[0].set_xlabel("Attendance %")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Attendance % Distribution (All Months)")
axes[0].legend()

month_att = master.groupby(["Month","Month_Num"])["Attendance_Pct"].mean().reset_index().sort_values("Month_Num")
axes[1].plot(month_att["Month"], month_att["Attendance_Pct"], marker='o', color="#2E75B6", linewidth=2)
axes[1].axhline(75, color="red", linestyle="--", linewidth=1, label="75% line")
axes[1].set_ylim(60, 105)
axes[1].set_title("Avg School Attendance by Month")
axes[1].set_ylabel("Attendance %")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.suptitle("Attendance Overview", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
"""))

cells.append(code("""# ── CHART 4: LWP Days by Employee (Annual) ────────────────────────────────────
lwp_annual = master.groupby("Employee_Name")["LWP"].sum().sort_values(ascending=False)
lwp_annual = lwp_annual[lwp_annual > 0]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#C00000" if v > 20 else "#ED7D31" if v > 10 else "#FFC000" for v in lwp_annual.values]
bars = ax.barh(lwp_annual.index, lwp_annual.values, color=colors)
ax.set_xlabel("Total LWP Days (Full Year)")
ax.set_title("Annual Leave Without Pay (LWP) — All Employees")
ax.axvline(10, color='orange', linestyle='--', linewidth=1, label="10-day warning")
ax.axvline(20, color='red', linestyle='--', linewidth=1, label="20-day critical")

for bar, val in zip(bars, lwp_annual.values):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f"{val:.1f}", va='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.show()
"""))

cells.append(code("""# ── CHART 5: Performance Score Distribution ────────────────────────────────────
perf_df = master.drop_duplicates("Employee_Name")[["Employee_Name","Performance_Score","Performance_Label","Designation"]]
perf_count = perf_df["Performance_Label"].value_counts().reindex(
    ["Excellent","Good","Average","Poor","Very Poor"]).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colors = ["#70AD47","#92D050","#FFC000","#ED7D31","#C00000"][:len(perf_count)]
axes[0].bar(perf_count.index, perf_count.values, color=colors)
axes[0].set_title("Performance Score Distribution")
axes[0].set_ylabel("No. of Employees")
for i, v in enumerate(perf_count.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

perf_desig = perf_df.groupby("Designation")["Performance_Score"].mean().sort_values(ascending=True)
axes[1].barh(perf_desig.index, perf_desig.values, color="#2E75B6")
axes[1].set_xlabel("Avg Performance Score (out of 5)")
axes[1].set_title("Avg Performance by Designation")
for i, v in enumerate(perf_desig.values):
    axes[1].text(v + 0.02, i, f"{v:.2f}", va='center')

plt.suptitle("Performance Overview", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
"""))

# ─── BUSINESS QUESTIONS ──────────────────────────────────────────────────────
cells.append(md("---\n## Step 5 — Business Questions & Insights"))

cells.append(md("### Q1 — Which employees have attendance below 75% in any month?"))
cells.append(code("""q1 = master[master["Attendance_Pct"] < 75][
    ["Employee_Name","Designation","Month","Year","Attendance_Pct","LWP"]
].sort_values("Attendance_Pct")

print(f"Employees with <75% attendance: {q1['Employee_Name'].nunique()} unique employees, {len(q1)} month-instances")
q1.reset_index(drop=True)
"""))
cells.append(code("""fig, ax = plt.subplots(figsize=(11, 5))
pivot_q1 = q1.groupby(["Employee_Name","Month"])["Attendance_Pct"].mean().unstack(fill_value=100)
sns.heatmap(pivot_q1, annot=True, fmt=".0f", cmap="RdYlGn", vmin=50, vmax=100,
            linewidths=0.5, ax=ax, cbar_kws={"label":"Attendance %"})
ax.set_title("Q1: Monthly Attendance Heatmap — Employees Who Fell Below 75% (At Least Once)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()
print("\\nInsight: Sheikh Ejaj had the most severe attendance drop (August: 40%).")
print("Sukhbeer Kaur fell below 75% in 3 separate months — a consistent pattern.")
"""))

cells.append(md("### Q2 — Who has the highest Late Mark + Early Mark combined?"))
cells.append(code("""q2 = master.groupby(["Employee_Name","Designation"]).agg(
    Total_Late=("Late_Mark","sum"),
    Total_Early=("Early_Mark","sum"),
    Total_LWP=("LWP","sum")
).reset_index()
q2["Total_Issues"] = q2["Total_Late"] + q2["Total_Early"]
q2 = q2[q2["Total_Issues"] > 0].sort_values("Total_Issues", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(q2))
ax.bar(x, q2["Total_Late"], label="Late Mark", color="#ED7D31")
ax.bar(x, q2["Total_Early"], bottom=q2["Total_Late"], label="Early Mark", color="#FFC000")
ax.set_xticks(x)
ax.set_xticklabels(q2["Employee_Name"], rotation=45, ha='right', fontsize=9)
ax.set_title("Q2: Total Punctuality Issues per Employee (Full Year)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()
print("\\nInsight: Wasim Jagirdar and Gagandeep Kaur Birdi had the most punctuality issues.")
print("Correlation between LWP and punctuality issues suggests a broader attendance attitude.")
"""))

cells.append(md("### Q3 — Which month had the worst overall school attendance?"))
cells.append(code("""q3 = master.groupby(["Month","Month_Num"]).agg(
    Avg_Attendance=("Attendance_Pct","mean"),
    Total_LWP=("LWP","sum"),
    Staff_Count=("Employee_Name","nunique")
).reset_index().sort_values("Month_Num")

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["#C00000" if v == q3["Avg_Attendance"].min() else
          "#70AD47" if v == q3["Avg_Attendance"].max() else "#2E75B6"
          for v in q3["Avg_Attendance"]]
bars = ax.bar(q3["Month"], q3["Avg_Attendance"], color=colors)
ax.set_ylim(70, 105)
ax.set_title("Q3: Average School Attendance by Month (2025-26)")
ax.set_ylabel("Avg Attendance %")
ax.axhline(75, color='red', linestyle='--', linewidth=1, label="75% threshold")
for bar, val in zip(bars, q3["Avg_Attendance"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f"{val:.1f}%",
            ha='center', fontsize=9, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

worst = q3.loc[q3["Avg_Attendance"].idxmin()]
print(f"\\nInsight: {worst['Month']} had the worst school-wide attendance at {worst['Avg_Attendance']:.1f}%")
"""))

cells.append(md("### Q4 — What is the monthly salary bill trend across the year?"))
cells.append(code("""q4 = master.groupby(["Month","Month_Num"]).agg(
    Total_Basic=("Basic_Salary","sum"),
    Total_Deductions=("LWP_Deduction","sum"),
    Total_Net=("Net_Salary_Payable","sum")
).reset_index().sort_values("Month_Num")

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(q4["Month"], q4["Total_Basic"], alpha=0.15, color="#2E75B6")
ax.plot(q4["Month"], q4["Total_Basic"], marker='s', label="Basic Salary Bill", color="#2E75B6", linewidth=2)
ax.plot(q4["Month"], q4["Total_Net"], marker='o', label="Net Payable (after LWP)", color="#70AD47", linewidth=2)
ax.fill_between(q4["Month"], q4["Total_Net"], q4["Total_Basic"], alpha=0.2, color="#C00000", label="LWP Deductions")
ax.set_title("Q4: Monthly Salary Bill — Basic vs Net Payable (2025-26)")
ax.set_ylabel("₹ Amount")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

total_deductions = q4["Total_Deductions"].sum()
print(f"\\nInsight: Total LWP deductions for the year = ₹{total_deductions:,.0f}")
print(f"Peak basic salary month: {q4.loc[q4['Total_Basic'].idxmax(),'Month']} (most staff present)")
"""))

cells.append(md("### Q5 — Top 5 costliest absentees by annual LWP deduction"))
cells.append(code("""q5 = master.groupby(["Employee_Name","Designation","Department"]).agg(
    Total_LWP=("LWP","sum"),
    Annual_LWP_Deduction=("LWP_Deduction","sum"),
    Basic_Salary=("Basic_Salary","first")
).reset_index().sort_values("Annual_LWP_Deduction", ascending=False).head(5)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(q5["Employee_Name"], q5["Annual_LWP_Deduction"],
               color=["#C00000","#ED7D31","#FFC000","#70AD47","#2E75B6"])
ax.set_xlabel("Annual LWP Deduction (₹)")
ax.set_title("Q5: Top 5 Employees — Highest Annual LWP Salary Deduction")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
for bar, (_, row) in zip(bars, q5.iterrows()):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f"₹{row['Annual_LWP_Deduction']:,.0f} ({row['Total_LWP']:.1f} days)",
            va='center', fontsize=9)
plt.tight_layout()
plt.show()
print(q5[["Employee_Name","Designation","Total_LWP","Annual_LWP_Deduction"]].to_string(index=False))
"""))

cells.append(md("### Q6 — Net vs Basic salary gap by designation"))
cells.append(code("""q6 = master.groupby("Designation").agg(
    Avg_Basic=("Basic_Salary","mean"),
    Avg_Net=("Net_Salary_Payable","mean"),
    Total_Deductions=("LWP_Deduction","sum")
).reset_index()
q6["Avg_Gap"] = q6["Avg_Basic"] - q6["Avg_Net"]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(q6))
w = 0.35
ax.bar(x - w/2, q6["Avg_Basic"], w, label="Avg Basic", color="#2E75B6")
ax.bar(x + w/2, q6["Avg_Net"], w, label="Avg Net Payable", color="#70AD47")
ax.set_xticks(x)
ax.set_xticklabels(q6["Designation"])
ax.set_title("Q6: Avg Basic vs Net Salary by Designation")
ax.set_ylabel("₹ Amount")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

print("Deduction impact by designation:")
for _, row in q6.iterrows():
    pct = (row["Avg_Gap"]/row["Avg_Basic"])*100
    print(f"  {row['Designation']:15s}: avg gap ₹{row['Avg_Gap']:,.0f}/month ({pct:.1f}% of basic)")
"""))

cells.append(md("### Q7 — CL vs LWP: Who exhausted CL and shifted to LWP?"))
cells.append(code("""q7 = master.groupby("Employee_Name").agg(
    Total_CL=("CL","sum"), Total_LWP=("LWP","sum")
).reset_index()
q7 = q7[q7["Total_LWP"] > 0].sort_values("Total_LWP", ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(q7))
ax.bar(x, q7["Total_CL"], label="Casual Leave (CL)", color="#2E75B6", alpha=0.85)
ax.bar(x, q7["Total_LWP"], bottom=q7["Total_CL"], label="Leave Without Pay (LWP)", color="#C00000", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(q7["Employee_Name"], rotation=45, ha='right', fontsize=9)
ax.set_title("Q7: CL Used vs LWP Days — Annual Total")
ax.set_ylabel("Days")
ax.legend()
plt.tight_layout()
plt.show()
print("\\nInsight: Employees with high LWP but low CL likely used unplanned leave.")
print("Recommendation: Introduce a planned leave policy with advance notice requirement.")
"""))

cells.append(md("### Q8 — Perfect attendance: Zero LWP all year"))
cells.append(code("""q8 = master.groupby(["Employee_Name","Designation","Department"]).agg(
    Total_LWP=("LWP","sum"),
    Avg_Attendance=("Attendance_Pct","mean"),
    Total_Late=("Late_Mark","sum"),
    Total_Early=("Early_Mark","sum")
).reset_index()
q8_perfect = q8[q8["Total_LWP"] == 0].sort_values("Avg_Attendance", ascending=False)

print(f"Employees with ZERO LWP all year: {len(q8_perfect)}")
print()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#70AD47" if att >= 95 else "#FFC000" for att in q8_perfect["Avg_Attendance"]]
bars = ax.barh(q8_perfect["Employee_Name"], q8_perfect["Avg_Attendance"], color=colors)
ax.axvline(95, color='green', linestyle='--', linewidth=1, label="95% excellence line")
ax.set_xlabel("Avg Attendance % (Full Year)")
ax.set_title("Q8: Perfect Attendance — Zero LWP Employees")
for bar, val in zip(bars, q8_perfect["Avg_Attendance"]):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.1f}%", va='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()
"""))

cells.append(md("### Q9 — Deteriorating attendance: Month-over-month trend"))
cells.append(code("""q9_pivot = master.pivot_table(index="Employee_Name", columns="Month_Num",
                              values="Attendance_Pct", aggfunc="mean")

# Calculate average month-over-month change
q9_pivot["MoM_Avg_Change"] = q9_pivot.diff(axis=1).mean(axis=1)
q9_deteriorating = q9_pivot["MoM_Avg_Change"].sort_values().head(8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of avg MoM change
colors = ["#C00000" if v < 0 else "#70AD47" for v in q9_deteriorating.values]
axes[0].barh(q9_deteriorating.index, q9_deteriorating.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title("Q9: Avg Month-over-Month Change in Attendance")
axes[0].set_xlabel("Avg MoM Change (%)")

# Right: line chart for top 3 most deteriorating
top3 = q9_deteriorating.head(3).index.tolist()
month_names = {v: k for k, v in MONTH_NUM.items()}
for emp in top3:
    row_data = q9_pivot.loc[emp, sorted([c for c in q9_pivot.columns if isinstance(c, int)])]
    axes[1].plot([month_names.get(m, m) for m in row_data.index],
                 row_data.values, marker='o', label=emp.split()[0], linewidth=1.5)
axes[1].set_title("Q9: Attendance Trend — Top 3 Declining Employees")
axes[1].set_ylabel("Attendance %")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
"""))

cells.append(md("### Q10 — Cost-cut analysis: Which designation tier to review?"))
cells.append(code("""q10 = master.groupby("Designation").agg(
    Headcount=("Employee_Name","nunique"),
    Total_Basic=("Basic_Salary","sum"),
    Total_Deductions=("LWP_Deduction","sum"),
    Total_Net=("Net_Salary_Payable","sum")
).reset_index()
q10["Deduction_Pct"] = (q10["Total_Deductions"] / q10["Total_Basic"] * 100).round(2)
q10["Target_10pct_Cut"] = (q10["Total_Basic"] * 0.10).round(0)
q10 = q10.sort_values("Deduction_Pct", ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(q10["Designation"], q10["Deduction_Pct"],
              color=["#C00000","#ED7D31","#FFC000","#70AD47"])
ax.set_title("Q10: LWP Deduction as % of Basic Salary — by Designation")
ax.set_ylabel("Deduction %")
for bar, val in zip(bars, q10["Deduction_Pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05, f"{val:.1f}%",
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print(q10[["Designation","Headcount","Total_Basic","Total_Deductions","Deduction_Pct"]].to_string(index=False))
"""))

cells.append(md("### Q11 — Top 3 highest-paid employees per department"))
cells.append(code("""emp_unique = master.drop_duplicates("Employee_Name")[["Employee_Name","Department","Designation","Basic_Salary"]]
emp_unique["Salary_Rank"] = emp_unique.groupby("Department")["Basic_Salary"].rank(method="dense", ascending=False)
q11 = emp_unique[emp_unique["Salary_Rank"] <= 3].sort_values(["Department","Salary_Rank"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, dept in enumerate(["Admin","Primary","Secondary"]):
    dept_df = q11[q11["Department"] == dept].sort_values("Basic_Salary", ascending=True)
    axes[i].barh(dept_df["Employee_Name"], dept_df["Basic_Salary"],
                 color=["#70AD47","#2E75B6","#FFC000"][:len(dept_df)])
    axes[i].set_title(f"{dept} Dept\nTop 3 Salaries")
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
    for bar, val in zip(axes[i].patches, dept_df["Basic_Salary"]):
        axes[i].text(val + 100, bar.get_y() + bar.get_height()/2,
                     f"₹{val:,}", va='center', fontsize=8)

plt.suptitle("Q11: Top 3 Highest Paid Employees per Department", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(q11[["Department","Salary_Rank","Employee_Name","Designation","Basic_Salary"]].to_string(index=False))
"""))

# ─── KEY FINDINGS ─────────────────────────────────────────────────────────────
cells.append(md("""---
## Step 6 — Key Findings & Recommendations

| # | Finding | Recommendation |
|---|---------|---------------|
| 1 | Sheikh Ejaj had 18.5 LWP days in August alone — highest single-month absenteeism | Review contract and introduce attendance incentive policy |
| 2 | Sukhbeer Kaur fell below 75% attendance in 3 separate months | Issue formal attendance warning; monitor closely |
| 3 | August was the worst month school-wide for attendance | Schedule staff reviews before August each year |
| 4 | Total annual LWP deductions cost the school significant salary savings | Introduce planned-leave system to reduce unplanned LWP |
| 5 | Jr. Teachers have the highest deduction % relative to their salary | Focus attendance incentives on this tier |
| 6 | 8+ employees maintained zero LWP — strong attendance culture exists | Recognise and reward these employees publicly |
| 7 | Maansingh Sardar (Principal) maintained perfect attendance all year | Leadership by example — positive signal for school culture |

---
*Project by Devraj Singh Sukhai | SDJEMS, Nanded | Stack: Python · MS SQL Server · Power BI*
"""))


In [12]:
# ─── BUILD NOTEBOOK ───────────────────────────────────────────────────────────
nb.cells = cells
output_path = r"D:\EMPLOYEE SCHOOL DATA ANALYSIS 2025-2026\SDJEMS_HR_Analysis_Built.ipynb"

# Add encoding="utf-8" here:
with open(output_path, "w", encoding="utf-8") as f:
    nbf.write(nb, f)